In [1]:

# STEP 2: Verify the Bronze Parquet files
# Import Python utilities for file and folder operations

from notebookutils import mssparkutils

# Location of our raw Wide World Importers data
bronze_path = "Files/wwi-raw-data"

# List the files available in the Bronze layer
bronze_files = mssparkutils.fs.ls(bronze_path)

print("BRONZE LAYER - SOURCE FILE INVENTORY")
print("-" * 45)

for file in bronze_files:
    print(file.name)

print("-" * 45)
print(f"Total source files: {len(bronze_files)}")


StatementMeta(, 24928b8a-bca5-46d0-a1ab-4dc76b886bea, 3, Finished, Available, Finished, False)

BRONZE LAYER - SOURCE FILE INVENTORY
---------------------------------------------
DimCity.parquet
DimCustomer.parquet
DimDate.parquet
DimEmployee.parquet
DimPaymentMethod.parquet
DimStockItem.parquet
DimSupplier.parquet
DimTransactionType.parquet
FactMovement.parquet
FactOrder.parquet
FactPurchase.parquet
FactSale.parquet
FactStockHolding.parquet
FactTransaction.parquet
---------------------------------------------
Total source files: 14


In [1]:

# STEP 4: Configure Bronze-to-Silver processing
# Customer is excluded because it has already been processed.

silver_config = {
    "DimCity": "silver_dim_city",
    "DimDate": "silver_dim_date",
    "DimEmployee": "silver_dim_employee",
    "DimPaymentMethod": "silver_dim_payment_method",
    "DimStockItem": "silver_dim_stock_item",
    "DimSupplier": "silver_dim_supplier",
    "DimTransactionType": "silver_dim_transaction_type",
    "FactMovement": "silver_fact_movement",
    "FactOrder": "silver_fact_order",
    "FactPurchase": "silver_fact_purchase",
    "FactSale": "silver_fact_sale",
    "FactStockHolding": "silver_fact_stock_holding",
    "FactTransaction": "silver_fact_transaction"
}


StatementMeta(, 1ae27ffb-6ded-44f4-92eb-77386b0000db, 3, Finished, Available, Finished, False)

In [2]:

# STEP 6: Dimension data quality rules
# These are expected keys; we'll verify them against source schemas.

dimension_keys = {
    "DimCity": "CityKey",
    "DimDate": "Date",
    "DimEmployee": "EmployeeKey",
    "DimPaymentMethod": "PaymentMethodKey",
    "DimStockItem": "StockItemKey",
    "DimSupplier": "SupplierKey",
    "DimTransactionType": "TransactionTypeKey"
}

def validate_dimension(df, source_name):
    key = dimension_keys[source_name]

    # Confirm the expected key exists
    if key not in df.columns:
        raise ValueError(
            f"{source_name}: Expected key '{key}' not found"
        )

    # Check null and duplicate keys
    summary = df.select(
        F.count("*").alias("total_rows"),
        F.countDistinct(key).alias("unique_keys"),
        F.sum(
            F.when(F.col(key).isNull(), 1).otherwise(0)
        ).alias("null_keys")
    ).first()

    null_count = summary["null_keys"] or 0
    duplicate_count = (
        summary["total_rows"]
        - summary["unique_keys"]
        - null_count
    )

    print(
        f"{source_name}: "
        f"Rows={summary['total_rows']}, "
        f"NullKeys={null_count}, "
        f"DuplicateKeys={duplicate_count}"
    )

    if null_count > 0 or duplicate_count > 0:
        raise ValueError(
            f"{source_name}: Data quality validation failed"
        )

    return True





StatementMeta(, 1ae27ffb-6ded-44f4-92eb-77386b0000db, 4, Finished, Available, Finished, False)

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

def transform_to_silver(source_name, target_name):

    # 1. Read the Bronze Parquet file
    source_path = f"Files/wwi-raw-data/{source_name}.parquet"
    df = spark.read.parquet(source_path)

    # 2. Trim leading/trailing spaces from text columns
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            df = df.withColumn(
                field.name,
                F.trim(F.col(f"`{field.name}`"))
            )

    # 3. Validate dimension keys before writing
    if source_name in dimension_keys:
        validate_dimension(df, source_name)

    # 4. Add audit timestamp
    df = df.withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )

    # 5. Write validated data to a Delta table
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_name)
    )

    print(f"SUCCESS: {source_name} -> {target_name}")


StatementMeta(, 1ae27ffb-6ded-44f4-92eb-77386b0000db, 6, Finished, Available, Finished, False)

In [3]:

# STEP 8: Test Silver transformation for DimCity only

transform_to_silver(
    source_name="DimCity",
    target_name="silver_dim_city"
)


StatementMeta(, 1ae27ffb-6ded-44f4-92eb-77386b0000db, 5, Finished, Available, Finished, False)

NameError: name 'transform_to_silver' is not defined

In [1]:

# STEP 9: Inspect FactSale schema only
# No full data scan and no Delta table write

df_sales = spark.read.parquet(
    "Files/wwi-raw-data/FactSale.parquet"
)

print("FACTSALE - COLUMN STRUCTURE")
print("-" * 45)

df_sales.printSchema()


StatementMeta(, abec5b74-fa01-4faf-bf23-a34f93f2feca, 3, Finished, Available, Finished, False)

FACTSALE - COLUMN STRUCTURE
---------------------------------------------
root
 |-- SaleKey: long (nullable = true)
 |-- CityKey: integer (nullable = true)
 |-- CustomerKey: integer (nullable = true)
 |-- BillToCustomerKey: integer (nullable = true)
 |-- StockItemKey: integer (nullable = true)
 |-- InvoiceDateKey: date (nullable = true)
 |-- DeliveryDateKey: date (nullable = true)
 |-- SalespersonKey: integer (nullable = true)
 |-- WWIInvoiceID: integer (nullable = true)
 |-- Description: string (nullable = true)
 |-- Package: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: decimal(18,2) (nullable = true)
 |-- TaxRate: decimal(18,3) (nullable = true)
 |-- TotalExcludingTax: decimal(18,2) (nullable = true)
 |-- TaxAmount: decimal(18,2) (nullable = true)
 |-- Profit: decimal(18,2) (nullable = true)
 |-- TotalIncludingTax: decimal(18,2) (nullable = true)
 |-- TotalDryItems: integer (nullable = true)
 |-- TotalChillerItems: integer (nullable = true)
 |-- L

In [2]:

# STEP 10: Define FactSale transformation
# Do not execute the function yet.

from pyspark.sql import functions as F

def transform_fact_sales():

    # Read Bronze
    df = spark.read.parquet(
        "Files/wwi-raw-data/FactSale.parquet"
    )

    # Clean descriptive text
    df = df.withColumn(
        "Description",
        F.trim(F.col("Description"))
    )

    # Add processing audit timestamp
    df = df.withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )

    # Validate the expected sales key
    summary = df.select(
        F.count("*").alias("total_rows"),
        F.countDistinct("SaleKey").alias("unique_keys"),
        F.count(
            F.when(F.col("SaleKey").isNull(), 1)
        ).alias("null_keys")
    ).first()

    duplicate_count = (
        summary["total_rows"]
        - summary["unique_keys"]
        - summary["null_keys"]
    )

    print(f"Rows: {summary['total_rows']}")
    print(f"Null SaleKeys: {summary['null_keys']}")
    print(f"Duplicate SaleKeys: {duplicate_count}")

    if summary["null_keys"] > 0 or duplicate_count > 0:
        raise ValueError(
            "FactSale validation failed. No table written."
        )

    # Write Silver Delta table
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("silver_fact_sale")
    )

    print("SUCCESS: FactSale -> silver_fact_sale")


StatementMeta(, abec5b74-fa01-4faf-bf23-a34f93f2feca, 4, Finished, Available, Finished, False)

In [3]:
# STEP 11: Execute FactSale Bronze-to-Silver transformation

transform_fact_sales()

StatementMeta(, abec5b74-fa01-4faf-bf23-a34f93f2feca, 5, Finished, Available, Finished, False)

Rows: 228265
Null SaleKeys: 0
Duplicate SaleKeys: 0
SUCCESS: FactSale -> silver_fact_sale


In [1]:

# STEP 12: Inspect DimDate schema
# Schema only - no full table scan or Delta write

df_date = spark.read.parquet(
    "Files/wwi-raw-data/DimDate.parquet"
)

print("DIMDATE - COLUMN STRUCTURE")
print("-" * 45)

df_date.printSchema()


StatementMeta(, 936fd08e-86b2-48f5-8fdc-cef5aacee14b, 3, Finished, Available, Finished, False)

DIMDATE - COLUMN STRUCTURE
---------------------------------------------
root
 |-- Date: date (nullable = true)
 |-- DayNumber: integer (nullable = true)
 |-- Day: string (nullable = true)
 |-- Month: string (nullable = true)
 |-- ShortMonth: string (nullable = true)
 |-- CalendarMonthNumber: integer (nullable = true)
 |-- CalendarMonthLabel: string (nullable = true)
 |-- CalendarYear: integer (nullable = true)
 |-- CalendarYearLabel: string (nullable = true)
 |-- FiscalMonthNumber: integer (nullable = true)
 |-- FiscalMonthLabel: string (nullable = true)
 |-- FiscalYear: integer (nullable = true)
 |-- FiscalYearLabel: string (nullable = true)
 |-- ISOWeekNumber: integer (nullable = true)



In [5]:

# STEP 13: Transform DimDate from Bronze to Silver

transform_to_silver(
    source_name="DimDate",
    target_name="silver_dim_date"
)


StatementMeta(, 1ae27ffb-6ded-44f4-92eb-77386b0000db, 7, Finished, Available, Finished, False)

DimDate: Rows=1461, NullKeys=0, DuplicateKeys=0
SUCCESS: DimDate -> silver_dim_date


In [6]:
# STEP 14: Inspect DimStockItem schema

df_stock = spark.read.parquet(
    "Files/wwi-raw-data/DimStockItem.parquet"
)

df_stock.printSchema()

StatementMeta(, 1ae27ffb-6ded-44f4-92eb-77386b0000db, 8, Finished, Available, Finished, False)

root
 |-- StockItemKey: integer (nullable = true)
 |-- WWIStockItemID: integer (nullable = true)
 |-- StockItem: string (nullable = true)
 |-- Color: string (nullable = true)
 |-- SellingPackage: string (nullable = true)
 |-- BuyingPackage: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- LeadTimeDays: integer (nullable = true)
 |-- QuantityPerOuter: integer (nullable = true)
 |-- IsChillerStock: boolean (nullable = true)
 |-- Barcode: string (nullable = true)
 |-- TaxRate: decimal(18,3) (nullable = true)
 |-- UnitPrice: decimal(18,2) (nullable = true)
 |-- RecommendedRetailPrice: decimal(18,2) (nullable = true)
 |-- TypicalWeightPerUnit: decimal(18,3) (nullable = true)
 |-- ValidFrom: timestamp (nullable = true)
 |-- ValidTo: timestamp (nullable = true)
 |-- LineageKey: integer (nullable = true)



In [7]:

# STEP 15: Transform DimStockItem from Bronze to Silver

transform_to_silver(
    source_name="DimStockItem",
    target_name="silver_dim_stock_item"
)


StatementMeta(, 1ae27ffb-6ded-44f4-92eb-77386b0000db, 9, Finished, Available, Finished, False)

DimStockItem: Rows=672, NullKeys=0, DuplicateKeys=0
SUCCESS: DimStockItem -> silver_dim_stock_item
